# Análisis exploratorio de datos: estructura, contenido y muestra analítica

## Introducción

El *Brazilian E-Commerce Public Dataset by Olist* reúne información de pedidos realizados en marketplaces de Brasil entre 2016 y 2018. Los datos se encuentran distribuidos en nueve tablas que describen clientes, vendedores, pedidos, ítems, productos, pagos, reseñas, categorías y referencias geográficas. La fuente original puede consultarse en [Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).

El objetivo del TFI es estudiar el aporte de la distancia geográfica y de las características físicas del producto a la predicción del costo de envío observado (`freight_value`). Antes de construir variables o ajustar modelos es necesario comprender cómo se organiza la información, cuál es la unidad de registro de cada tabla y qué relaciones permiten formar una observación analítica.

Este notebook presenta una primera lectura descriptiva del dataset. El recorrido comienza por el modelo de datos, continúa con el contenido y la calidad de cada tabla, resume las principales magnitudes del comercio electrónico y finaliza con la construcción de la muestra que se utilizará en las etapas posteriores.

## Preparación del entorno

La siguiente celda instala las dependencias declaradas en `requirements.txt` utilizando el mismo intérprete de Python que ejecuta el kernel de Jupyter. De este modo, el entorno puede reproducirse antes de comenzar el análisis.

In [1]:
# Instalacion de dependencias del proyecto
from pathlib import Path
import subprocess
import sys

candidatos_requirements = [
    Path.cwd() / "requirements.txt",
    Path.cwd().parent / "requirements.txt",
    Path.cwd() / "TFI_ITBA" / "requirements.txt",
]
REQUIREMENTS = next(
    (ruta for ruta in candidatos_requirements if ruta.exists()),
    None,
)

if REQUIREMENTS is None:
    raise FileNotFoundError("No se encontró requirements.txt.")

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REQUIREMENTS)]
)
print(f"Dependencias verificadas desde: {REQUIREMENTS.resolve()}")

Dependencias verificadas desde: C:\Users\daniel.fernandez\Desktop\Repos\TFI_ITBA\requirements.txt


## Objetivos del análisis exploratorio

1. Describir las tablas, sus unidades de registro y sus relaciones.
2. Identificar los atributos disponibles, sus tipos de datos y sus valores faltantes.
3. Cuantificar pedidos, clientes, vendedores, productos y categorías.
4. Examinar la distribución temporal de los pedidos y la variable objetivo `freight_value`.
5. Revisar las variables físicas que posteriormente se utilizarán como predictores.
6. Construir de manera trazable la muestra de pedidos entregados con un único ítem.
7. Documentar las exclusiones necesarias antes de calcular la distancia geográfica.

In [2]:
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

# Permite ejecutar el notebook desde Repos, TFI_ITBA o notebooks/.
candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd() / "TFI_ITBA"]
PROYECTO = next((ruta for ruta in candidatos if (ruta / "data" / "parquet").exists()), None)
if PROYECTO is None:
    raise FileNotFoundError("No se encontró la carpeta data/parquet del proyecto.")

DATA_DIR = PROYECTO / "data" / "parquet"

archivos = {
    "customers": "olist_customers_dataset.parquet",
    "orders": "olist_orders_dataset.parquet",
    "order_items": "olist_order_items_dataset.parquet",
    "products": "olist_products_dataset.parquet",
    "sellers": "olist_sellers_dataset.parquet",
    "geolocation": "olist_geolocation_dataset.parquet",
    "order_payments": "olist_order_payments_dataset.parquet",
    "order_reviews": "olist_order_reviews_dataset.parquet",
    "category_translation": "product_category_name_translation.parquet",
}

tablas = {nombre: pd.read_parquet(DATA_DIR / archivo) for nombre, archivo in archivos.items()}

# Criterios visuales comunes a todos los gráficos del notebook.
COLOR_PRINCIPAL = "#1F4E79"
COLOR_SECUNDARIO = "#4E79A7"
COLOR_ACENTO = "#D97706"
COLOR_VERDE = "#2F7D55"
COLOR_TEXTO = "#25313C"
COLOR_REJILLA = "#E5E9EF"


def numero_es(valor):
    """Representa enteros con punto como separador de miles."""
    return f"{int(valor):,}".replace(",", ".")


def estilo_profesional(grafico):
    """Aplica una presentación homogénea sin modificar los datos del gráfico."""
    return (
        grafico
        .configure_view(stroke=None)
        .configure_axis(
            labelFont="Arial",
            titleFont="Arial",
            labelFontSize=11,
            titleFontSize=12,
            labelColor=COLOR_TEXTO,
            titleColor=COLOR_TEXTO,
            gridColor=COLOR_REJILLA,
            gridOpacity=0.8,
            domainColor="#A7B0BA",
            tickColor="#A7B0BA",
            labelPadding=6,
            titlePadding=10,
        )
        .configure_title(
            font="Arial",
            fontSize=17,
            fontWeight=600,
            color=COLOR_TEXTO,
            anchor="start",
            offset=14,
            subtitleFont="Arial",
            subtitleFontSize=11,
            subtitleColor="#5F6B76",
            subtitlePadding=7,
        )
        .configure_legend(
            labelFont="Arial",
            titleFont="Arial",
            labelFontSize=10,
            titleFontSize=11,
            labelColor=COLOR_TEXTO,
            titleColor=COLOR_TEXTO,
        )
    )


print(f"Carpeta de datos: {DATA_DIR.resolve()}")
print(f"Tablas cargadas: {len(tablas)}")
print(f"Versión de Altair: {alt.__version__}")

Carpeta de datos: C:\Users\daniel.fernandez\Desktop\Repos\TFI_ITBA\data\parquet
Tablas cargadas: 9
Versión de Altair: 6.3.0


# 1. Modelo de datos

## 1.1. Catálogo de tablas

Cada tabla representa una unidad de registro diferente. Reconocer esa granularidad es indispensable para evitar duplicaciones al realizar uniones. Por ejemplo, un pedido puede contener varios ítems y varios pagos; por lo tanto, unir ambas tablas sin una regla previa puede multiplicar artificialmente las filas.

In [3]:
catalogo_tablas = pd.DataFrame(
    [
        ["customers", "Registro de cliente asociado a un pedido", "customer_id", "Identificación del cliente, ciudad, estado y prefijo postal"],
        ["orders", "Un pedido", "order_id", "Estado del pedido y fechas principales del proceso de compra y entrega"],
        ["order_items", "Un ítem dentro de un pedido", "order_id + order_item_id", "Producto, vendedor, precio y costo de envío del ítem"],
        ["products", "Un producto", "product_id", "Categoría y características físicas del producto"],
        ["sellers", "Un vendedor", "seller_id", "Ciudad, estado y prefijo postal del vendedor"],
        ["geolocation", "Una coordenada observada para un prefijo postal", "Sin clave única en la tabla original", "Latitud, longitud, ciudad y estado asociados al prefijo postal"],
        ["order_payments", "Un pago o cuota de pago de un pedido", "order_id + payment_sequential", "Medio de pago, cuotas e importe abonado"],
        ["order_reviews", "Una reseña asociada a un pedido", "review_id", "Puntuación, comentarios y fechas de la reseña"],
        ["category_translation", "Una traducción de categoría", "product_category_name", "Correspondencia entre el nombre de categoría en portugués e inglés"],
    ],
    columns=["tabla", "unidad_de_registro", "clave", "descripcion"],
)

display(catalogo_tablas)

,tabla,unidad_de_registro,clave,descripcion
0,customers,Registro de cliente asociado a un pedido,customer_id,"Identificación del cliente, ciudad, estado y p..."
1,orders,Un pedido,order_id,Estado del pedido y fechas principales del pro...
2,order_items,Un ítem dentro de un pedido,order_id + order_item_id,"Producto, vendedor, precio y costo de envío de..."
3,products,Un producto,product_id,Categoría y características físicas del producto
4,sellers,Un vendedor,seller_id,"Ciudad, estado y prefijo postal del vendedor"
5,geolocation,Una coordenada observada para un prefijo postal,Sin clave única en la tabla original,"Latitud, longitud, ciudad y estado asociados a..."
6,order_payments,Un pago o cuota de pago de un pedido,order_id + payment_sequential,"Medio de pago, cuotas e importe abonado"
7,order_reviews,Una reseña asociada a un pedido,review_id,"Puntuación, comentarios y fechas de la reseña"
8,category_translation,Una traducción de categoría,product_category_name,Correspondencia entre el nombre de categoría e...


## 1.2. Relaciones entre las tablas

El diagrama presenta las columnas utilizadas para vincular las nueve tablas. Las conexiones de `customers` y `sellers` con `geolocation` se muestran mediante el prefijo postal. En este notebook, `geolocation` se resume previamente a una fila por prefijo para construir esas uniones.

In [4]:
nodos = pd.DataFrame(
    [
        ["geolocation", 0.0, 3.0, "Referencia geográfica", "Unión*: geolocation_zip_code_prefix", "", "Referencia"],
        ["customers", 3.2, 5.4, "Cliente asociado al pedido", "ID: customer_id", "Unión: customer_zip_code_prefix", "Dimensión"],
        ["sellers", 3.2, 0.6, "Vendedor", "ID: seller_id", "Unión: seller_zip_code_prefix", "Dimensión"],
        ["orders", 6.4, 5.4, "Pedido", "ID: order_id", "Unión: customer_id", "Tabla central"],
        ["order_reviews", 9.6, 7.3, "Reseña del pedido", "ID: review_id", "Unión: order_id", "Transacción auxiliar"],
        ["order_payments", 9.6, 5.4, "Pago del pedido", "ID: order_id + payment_sequential", "", "Transacción auxiliar"],
        ["order_items", 9.6, 2.5, "Ítem del pedido", "ID: order_id + order_item_id", "Unión: product_id · seller_id", "Tabla central"],
        ["products", 12.8, 2.5, "Producto", "ID: product_id", "Unión: product_category_name", "Dimensión"],
        ["category_translation", 16.0, 2.5, "Traducción de categoría", "ID: product_category_name", "", "Referencia"],
    ],
    columns=["tabla", "x", "y", "unidad", "campo_1", "campo_2", "tipo"],
)

ancho_nodo = 2.7
alto_nodo = 1.35
nodos["x2"] = nodos["x"] + ancho_nodo
nodos["y2"] = nodos["y"] + alto_nodo
nodos["xc"] = nodos["x"] + ancho_nodo / 2
nodos["y_nombre"] = nodos["y"] + 1.02
nodos["y_unidad"] = nodos["y"] + 0.72
nodos["y_campo_1"] = nodos["y"] + 0.41
nodos["y_campo_2"] = nodos["y"] + 0.17

relaciones = pd.DataFrame(
    [
        [2.70, 3.68, 3.20, 6.08],
        [2.70, 3.48, 3.20, 1.28],
        [5.90, 6.08, 6.40, 6.08],
        [9.10, 6.08, 9.60, 7.98],
        [9.10, 6.08, 9.60, 6.08],
        [9.10, 5.85, 9.60, 3.18],
        [5.90, 1.28, 9.60, 3.18],
        [12.30, 3.18, 12.80, 3.18],
        [15.50, 3.18, 16.00, 3.18],
    ],
    columns=["x", "y", "x2", "y2"],
)

escala_x = alt.Scale(domain=[-0.3, 19.1])
escala_y = alt.Scale(domain=[-0.2, 9.0])

lineas = alt.Chart(relaciones).mark_rule(
    color="#7B8794", strokeWidth=1.8
).encode(
    x=alt.X("x:Q", scale=escala_x, axis=None),
    x2="x2:Q",
    y=alt.Y("y:Q", scale=escala_y, axis=None),
    y2="y2:Q",
)

cajas = alt.Chart(nodos).mark_rect(
    stroke="#52606D", strokeWidth=1.1, cornerRadius=5
).encode(
    x=alt.X("x:Q", scale=escala_x, axis=None),
    x2="x2:Q",
    y=alt.Y("y:Q", scale=escala_y, axis=None),
    y2="y2:Q",
    color=alt.Color(
        "tipo:N",
        scale=alt.Scale(
            domain=["Tabla central", "Dimensión", "Transacción auxiliar", "Referencia"],
            range=["#1F4E79", "#DCE6F1", "#FCE8D5", "#E8ECEF"],
        ),
        legend=alt.Legend(title=None, orient="bottom", direction="horizontal"),
    ),
    tooltip=["tabla:N", "unidad:N", "campo_1:N", "campo_2:N", "tipo:N"],
)

nombres = alt.Chart(nodos).mark_text(
    font="Arial", fontSize=11, fontWeight="bold"
).encode(
    x=alt.X("xc:Q", scale=escala_x, axis=None),
    y=alt.Y("y_nombre:Q", scale=escala_y, axis=None),
    text="tabla:N",
    color=alt.condition(alt.datum.tipo == "Tabla central", alt.value("white"), alt.value(COLOR_TEXTO)),
)

unidades = alt.Chart(nodos).mark_text(font="Arial", fontSize=9).encode(
    x=alt.X("xc:Q", scale=escala_x, axis=None),
    y=alt.Y("y_unidad:Q", scale=escala_y, axis=None),
    text="unidad:N",
    color=alt.condition(alt.datum.tipo == "Tabla central", alt.value("#EAF1F7"), alt.value("#4B5563")),
)

campos_1 = alt.Chart(nodos).mark_text(font="Arial", fontSize=8.2).encode(
    x=alt.X("xc:Q", scale=escala_x, axis=None),
    y=alt.Y("y_campo_1:Q", scale=escala_y, axis=None),
    text="campo_1:N",
    color=alt.condition(alt.datum.tipo == "Tabla central", alt.value("#EAF1F7"), alt.value("#4B5563")),
)

campos_2 = alt.Chart(nodos).mark_text(font="Arial", fontSize=8.2).encode(
    x=alt.X("xc:Q", scale=escala_x, axis=None),
    y=alt.Y("y_campo_2:Q", scale=escala_y, axis=None),
    text="campo_2:N",
    color=alt.condition(alt.datum.tipo == "Tabla central", alt.value("#EAF1F7"), alt.value("#4B5563")),
)

diagrama_modelo = (
    lineas + cajas + nombres + unidades + campos_1 + campos_2
).properties(
    width=1080,
    height=520,
    title=alt.TitleParams(
        text="Modelo lógico simplificado del dataset",
        subtitle="Las líneas representan vínculos mediante los campos indicados dentro de cada tabla",
    ),
)

estilo_profesional(diagrama_modelo)

alt.LayerChart(...)

### Lectura del modelo

- `orders` se vincula con `customers` mediante `customer_id`.
- `order_items` contiene las referencias a `order_id`, `product_id` y `seller_id`.
- `products` se vincula con `category_translation` mediante `product_category_name`.
- `order_payments` y `order_reviews` se vinculan con `orders` mediante `order_id`.
- `customers` y `sellers` contienen los prefijos postales utilizados para incorporar las coordenadas resumidas de `geolocation`.

# 2. Dimensiones e información de las tablas

## 2.1. Cantidad de filas y columnas

Se presentan las dimensiones observadas en cada archivo. La escala logarítmica del gráfico permite comparar tablas cuyos números de registros se encuentran en órdenes de magnitud diferentes.

In [5]:
resumen_dimensiones = pd.DataFrame(
    [
        {"tabla": nombre, "filas": len(df), "columnas": df.shape[1]}
        for nombre, df in tablas.items()
    ]
).sort_values("filas", ascending=False)
resumen_dimensiones["filas_texto"] = resumen_dimensiones["filas"].map(numero_es)
resumen_dimensiones["orden"] = range(1, len(resumen_dimensiones) + 1)

display(resumen_dimensiones[["tabla", "filas", "columnas"]])

resumen_dimensiones["inicio_log"] = 50

lineas_tablas = alt.Chart(resumen_dimensiones).mark_rule(
    color="#B8C6D3", strokeWidth=3
).encode(
    x=alt.X(
        "inicio_log:Q",
        title="Cantidad de filas (escala logarítmica)",
        scale=alt.Scale(type="log", domain=[50, 1_500_000]),
        axis=alt.Axis(format="~s", grid=True, tickCount=6),
    ),
    x2="filas:Q",
    y=alt.Y("tabla:N", sort=alt.SortField("orden", order="ascending"), title=None, axis=alt.Axis(labelLimit=220)),
)
puntos_tablas = alt.Chart(resumen_dimensiones).mark_point(
    filled=True, color=COLOR_PRINCIPAL, size=90, stroke="white", strokeWidth=1
).encode(
    x=alt.X(
        "filas:Q",
        title="Cantidad de filas (escala logarítmica)",
        scale=alt.Scale(type="log", domain=[50, 1_500_000]),
        axis=alt.Axis(format="~s", grid=True, tickCount=6),
    ),
    y=alt.Y("tabla:N", sort=alt.SortField("orden", order="ascending"), title=None, axis=alt.Axis(labelLimit=220)),
    tooltip=["tabla:N", "filas_texto:N", "columnas:Q"],
)
etiquetas_tablas = alt.Chart(resumen_dimensiones).mark_text(
    align="left", dx=7, font="Arial", fontSize=10, color=COLOR_TEXTO
).encode(
    x=alt.X("filas:Q", scale=alt.Scale(type="log", domain=[50, 1_500_000])),
    y=alt.Y("tabla:N", sort=alt.SortField("orden", order="ascending")),
    text="filas_texto:N",
)

grafico_dimensiones = (lineas_tablas + puntos_tablas + etiquetas_tablas).properties(
    width=760,
    height=340,
    title=alt.TitleParams(
        text="Cantidad de registros por tabla",
        subtitle="Cada punto muestra el valor exacto; el eje horizontal utiliza escala logarítmica",
    ),
)
estilo_profesional(grafico_dimensiones)

,tabla,filas,columnas
5,geolocation,1000163,5
2,order_items,112650,7
6,order_payments,103886,5
0,customers,99441,5
1,orders,99441,8
7,order_reviews,99224,7
3,products,32951,9
4,sellers,3095,4
8,category_translation,71,2


alt.LayerChart(...)

## 2.2. Estructura de cada tabla (`info`)

La salida de `info()` permite observar los nombres de las columnas, sus tipos de datos, la cantidad de valores no nulos y el uso aproximado de memoria. En los archivos originales, las fechas se encuentran almacenadas como texto y se convertirán explícitamente cuando sean necesarias.

In [6]:
for nombre, df in tablas.items():
    print("\n" + "=" * 80)
    print(f"TABLA: {nombre}")
    print("=" * 80)
    df.info(show_counts=True)


TABLA: customers
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB

TABLA: orders
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestam

## 2.3. Diccionario de atributos

El diccionario combina la descripción conceptual de cada atributo con información calculada directamente desde los archivos: tipo de dato, cantidad de valores nulos y cantidad de valores distintos.

In [7]:
descripciones = {
    "customers": {
        "customer_id": "Identificador del registro de cliente asociado al pedido",
        "customer_unique_id": "Identificador persistente que permite reconocer compras del mismo cliente",
        "customer_zip_code_prefix": "Prefijo de cinco posiciones del código postal del cliente",
        "customer_city": "Ciudad del cliente",
        "customer_state": "Estado del cliente",
    },
    "orders": {
        "order_id": "Identificador del pedido",
        "customer_id": "Identificador que vincula el pedido con customers",
        "order_status": "Estado del pedido",
        "order_purchase_timestamp": "Fecha y hora de realización de la compra",
        "order_approved_at": "Fecha y hora de aprobación del pago",
        "order_delivered_carrier_date": "Fecha y hora de entrega al transportista",
        "order_delivered_customer_date": "Fecha y hora de entrega al cliente",
        "order_estimated_delivery_date": "Fecha estimada de entrega informada al cliente",
    },
    "order_items": {
        "order_id": "Identificador del pedido",
        "order_item_id": "Número secuencial del ítem dentro del pedido",
        "product_id": "Identificador del producto",
        "seller_id": "Identificador del vendedor",
        "shipping_limit_date": "Fecha límite para que el vendedor despache el ítem",
        "price": "Precio del producto en la operación",
        "freight_value": "Costo de envío asignado al ítem; variable objetivo del TFI",
    },
    "products": {
        "product_id": "Identificador del producto",
        "product_category_name": "Categoría del producto en portugués",
        "product_name_lenght": "Cantidad de caracteres del nombre del producto",
        "product_description_lenght": "Cantidad de caracteres de la descripción",
        "product_photos_qty": "Cantidad de fotografías publicadas",
        "product_weight_g": "Peso del producto en gramos",
        "product_length_cm": "Largo del producto en centímetros",
        "product_height_cm": "Alto del producto en centímetros",
        "product_width_cm": "Ancho del producto en centímetros",
    },
    "sellers": {
        "seller_id": "Identificador del vendedor",
        "seller_zip_code_prefix": "Prefijo de cinco posiciones del código postal del vendedor",
        "seller_city": "Ciudad del vendedor",
        "seller_state": "Estado del vendedor",
    },
    "geolocation": {
        "geolocation_zip_code_prefix": "Prefijo de cinco posiciones del código postal",
        "geolocation_lat": "Latitud de la observación geográfica",
        "geolocation_lng": "Longitud de la observación geográfica",
        "geolocation_city": "Ciudad informada para la observación",
        "geolocation_state": "Estado informado para la observación",
    },
    "order_payments": {
        "order_id": "Identificador del pedido",
        "payment_sequential": "Número secuencial del pago dentro del pedido",
        "payment_type": "Medio de pago",
        "payment_installments": "Cantidad de cuotas",
        "payment_value": "Importe del pago",
    },
    "order_reviews": {
        "review_id": "Identificador de la reseña",
        "order_id": "Identificador del pedido reseñado",
        "review_score": "Puntuación otorgada por el cliente",
        "review_comment_title": "Título del comentario",
        "review_comment_message": "Texto del comentario",
        "review_creation_date": "Fecha de creación de la reseña",
        "review_answer_timestamp": "Fecha y hora de respuesta de la reseña",
    },
    "category_translation": {
        "product_category_name": "Nombre de la categoría en portugués",
        "product_category_name_english": "Traducción de la categoría al inglés",
    },
}

filas_diccionario = []
for nombre, df in tablas.items():
    for columna in df.columns:
        filas_diccionario.append(
            {
                "tabla": nombre,
                "atributo": columna,
                "tipo": str(df[columna].dtype),
                "nulos": int(df[columna].isna().sum()),
                "valores_distintos": int(df[columna].nunique(dropna=True)),
                "descripcion": descripciones[nombre][columna],
            }
        )

diccionario_atributos = pd.DataFrame(filas_diccionario)

for nombre in archivos:
    print(f"\nTabla: {nombre}")
    display(
        diccionario_atributos.loc[
            diccionario_atributos["tabla"].eq(nombre),
            ["atributo", "tipo", "nulos", "valores_distintos", "descripcion"],
        ].reset_index(drop=True)
    )


Tabla: customers


,atributo,tipo,nulos,valores_distintos,descripcion
0,customer_id,object,0,99441,Identificador del registro de cliente asociado...
1,customer_unique_id,object,0,96096,Identificador persistente que permite reconoce...
2,customer_zip_code_prefix,int64,0,14994,Prefijo de cinco posiciones del código postal ...
3,customer_city,object,0,4119,Ciudad del cliente
4,customer_state,object,0,27,Estado del cliente



Tabla: orders


,atributo,tipo,nulos,valores_distintos,descripcion
0,order_id,object,0,99441,Identificador del pedido
1,customer_id,object,0,99441,Identificador que vincula el pedido con customers
2,order_status,object,0,8,Estado del pedido
3,order_purchase_timestamp,object,0,98875,Fecha y hora de realización de la compra
4,order_approved_at,object,160,90733,Fecha y hora de aprobación del pago
5,order_delivered_carrier_date,object,1783,81018,Fecha y hora de entrega al transportista
6,order_delivered_customer_date,object,2965,95664,Fecha y hora de entrega al cliente
7,order_estimated_delivery_date,object,0,459,Fecha estimada de entrega informada al cliente



Tabla: order_items


,atributo,tipo,nulos,valores_distintos,descripcion
0,order_id,object,0,98666,Identificador del pedido
1,order_item_id,int64,0,21,Número secuencial del ítem dentro del pedido
2,product_id,object,0,32951,Identificador del producto
3,seller_id,object,0,3095,Identificador del vendedor
4,shipping_limit_date,object,0,93318,Fecha límite para que el vendedor despache el ...
5,price,float64,0,5968,Precio del producto en la operación
6,freight_value,float64,0,6999,Costo de envío asignado al ítem; variable obje...


... (8 outputs left out)


Tabla: order_reviews


,atributo,tipo,nulos,valores_distintos,descripcion
0,review_id,object,0,98410,Identificador de la reseña
1,order_id,object,0,98673,Identificador del pedido reseñado
2,review_score,int64,0,5,Puntuación otorgada por el cliente
3,review_comment_title,object,87656,4527,Título del comentario
4,review_comment_message,object,58247,36159,Texto del comentario
5,review_creation_date,object,0,636,Fecha de creación de la reseña
6,review_answer_timestamp,object,0,98248,Fecha y hora de respuesta de la reseña



Tabla: category_translation


,atributo,tipo,nulos,valores_distintos,descripcion
0,product_category_name,object,0,71,Nombre de la categoría en portugués
1,product_category_name_english,object,0,71,Traducción de la categoría al inglés


# 3. Calidad general de los datos

Para cada atributo con valores faltantes se informa la cantidad absoluta y el porcentaje respecto de las filas de su tabla. Esta comparación conserva la unidad de registro de cada fuente.

In [8]:
faltantes = diccionario_atributos.loc[
    diccionario_atributos["nulos"].gt(0), ["tabla", "atributo", "nulos"]
].copy()
filas_por_tabla = {nombre: len(df) for nombre, df in tablas.items()}
faltantes["filas_tabla"] = faltantes["tabla"].map(filas_por_tabla)
faltantes["porcentaje_nulos"] = 100 * faltantes["nulos"] / faltantes["filas_tabla"]
faltantes["tabla_atributo"] = faltantes["tabla"] + " · " + faltantes["atributo"]
faltantes["nulos_texto"] = faltantes["nulos"].map(numero_es)
faltantes["porcentaje_texto"] = faltantes["porcentaje_nulos"].map(lambda x: f"{x:.2f}%")
faltantes = faltantes.sort_values("porcentaje_nulos", ascending=False)

display(faltantes[["tabla", "atributo", "nulos", "porcentaje_nulos"]])

barras_faltantes = alt.Chart(faltantes).mark_bar(
    color=COLOR_ACENTO, cornerRadiusEnd=3, size=16
).encode(
    x=alt.X(
        "porcentaje_nulos:Q",
        title="Valores faltantes sobre las filas de la tabla (%)",
        axis=alt.Axis(format=".0f"),
    ),
    y=alt.Y(
        "tabla_atributo:N",
        sort="-x",
        title=None,
        axis=alt.Axis(labelLimit=320, labelFontSize=10),
    ),
    tooltip=[
        "tabla:N",
        "atributo:N",
        alt.Tooltip("nulos_texto:N", title="Valores faltantes"),
        alt.Tooltip("porcentaje_texto:N", title="Porcentaje"),
    ],
)

grafico_faltantes = barras_faltantes.properties(
    width=700,
    height=max(380, 22 * len(faltantes)),
    title=alt.TitleParams(
        text="Proporción de valores faltantes por atributo",
        subtitle="Se incluyen únicamente los atributos que registran al menos un valor faltante",
    ),
)
estilo_profesional(grafico_faltantes)

,tabla,atributo,nulos,porcentaje_nulos
46,order_reviews,review_comment_title,87656,88.341530
47,order_reviews,review_comment_message,58247,58.702532
11,orders,order_delivered_customer_date,2965,2.981668
22,products,product_name_lenght,610,1.851234
21,products,product_category_name,610,1.851234
23,products,product_description_lenght,610,1.851234
24,products,product_photos_qty,610,1.851234
10,orders,order_delivered_carrier_date,1783,1.793023
9,orders,order_approved_at,160,0.160899
26,products,product_length_cm,2,0.006070


alt.Chart(...)

# 4. Magnitudes principales del dataset

Los siguientes conteos describen el tamaño del conjunto de datos. Los clientes diferentes se contabilizan mediante `customer_unique_id`; los registros asociados a cada pedido se identifican mediante `customer_id`.

In [9]:
customers = tablas["customers"]
orders = tablas["orders"].copy()
order_items = tablas["order_items"]
products = tablas["products"].copy()
sellers = tablas["sellers"]

indicadores = pd.DataFrame(
    {
        "indicador": [
            "Pedidos",
            "Clientes únicos",
            "Ítems de pedido",
            "Productos",
            "Vendedores",
            "Categorías de producto",
        ],
        "cantidad": [
            orders["order_id"].nunique(),
            customers["customer_unique_id"].nunique(),
            len(order_items),
            products["product_id"].nunique(),
            sellers["seller_id"].nunique(),
            products["product_category_name"].nunique(),
        ],
    }
)
display(indicadores)

,indicador,cantidad
0,Pedidos,99441
1,Clientes únicos,96096
2,Ítems de pedido,112650
3,Productos,32951
4,Vendedores,3095
5,Categorías de producto,73


## 4.1. Productos y categorías

La tabla `products` contiene una fila por `product_id`. El gráfico cuenta productos distintos por categoría y no representa la frecuencia de venta de esos productos.

In [10]:
productos_por_categoria = (
    products.dropna(subset=["product_category_name"])
    .groupby("product_category_name", as_index=False)["product_id"]
    .nunique()
    .rename(columns={"product_id": "cantidad_productos"})
    .sort_values("cantidad_productos", ascending=False)
    .head(15)
)
productos_por_categoria["categoria"] = productos_por_categoria["product_category_name"].str.replace("_", " ", regex=False)
productos_por_categoria["cantidad_texto"] = productos_por_categoria["cantidad_productos"].map(numero_es)

display(productos_por_categoria[["product_category_name", "cantidad_productos"]])

barras_categorias = alt.Chart(productos_por_categoria).mark_bar(
    color=COLOR_VERDE, cornerRadiusEnd=4, size=19
).encode(
    x=alt.X("cantidad_productos:Q", title="Cantidad de productos distintos", axis=alt.Axis(grid=True)),
    y=alt.Y(
        "categoria:N",
        sort="-x",
        title=None,
        axis=alt.Axis(labelLimit=260, labelFontSize=10),
    ),
    tooltip=[
        alt.Tooltip("product_category_name:N", title="Categoría original"),
        alt.Tooltip("cantidad_texto:N", title="Productos"),
    ],
)
etiquetas_categorias = alt.Chart(productos_por_categoria).mark_text(
    align="left", dx=6, font="Arial", fontSize=10, color=COLOR_TEXTO
).encode(
    x="cantidad_productos:Q",
    y=alt.Y("categoria:N", sort="-x"),
    text="cantidad_texto:N",
)

grafico_categorias = (barras_categorias + etiquetas_categorias).properties(
    width=700,
    height=430,
    title=alt.TitleParams(
        text="Categorías con mayor cantidad de productos",
        subtitle="Se muestran las primeras 15 categorías según productos distintos",
    ),
)
estilo_profesional(grafico_categorias)

,product_category_name,cantidad_productos
13,cama_mesa_banho,3029
32,esporte_lazer,2867
54,moveis_decoracao,2657
11,beleza_saude,2444
72,utilidades_domesticas,2335
8,automotivo,1900
44,informatica_acessorios,1639
12,brinquedos,1411
66,relogios_presentes,1329
70,telefonia,1134


alt.LayerChart(...)

## 4.2. Estado y evolución temporal de los pedidos

Se describen la cantidad de pedidos por estado y la cantidad mensual según `order_purchase_timestamp`. La construcción de la muestra de la sección 6 conserva únicamente los pedidos cuyo estado es `delivered`.

In [11]:
pedidos_por_estado = (
    orders["order_status"].value_counts()
    .rename_axis("estado_pedido")
    .reset_index(name="cantidad_pedidos")
)
pedidos_por_estado["cantidad_texto"] = pedidos_por_estado["cantidad_pedidos"].map(numero_es)

orders["fecha_compra"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
pedidos_por_mes = (
    orders.dropna(subset=["fecha_compra"])
    .assign(mes=lambda x: x["fecha_compra"].dt.to_period("M").dt.to_timestamp())
    .groupby("mes", as_index=False)["order_id"]
    .nunique()
    .rename(columns={"order_id": "cantidad_pedidos"})
)
pedidos_por_mes["cantidad_texto"] = pedidos_por_mes["cantidad_pedidos"].map(numero_es)

barras_estados = alt.Chart(pedidos_por_estado).mark_bar(
    color=COLOR_SECUNDARIO, cornerRadiusEnd=3, size=17
).encode(
    x=alt.X("cantidad_pedidos:Q", title="Cantidad de pedidos"),
    y=alt.Y("estado_pedido:N", sort="-x", title=None),
    tooltip=["estado_pedido:N", alt.Tooltip("cantidad_texto:N", title="Pedidos")],
)
texto_estados = alt.Chart(pedidos_por_estado).mark_text(
    align="left", dx=5, font="Arial", fontSize=9, color=COLOR_TEXTO
).encode(
    x="cantidad_pedidos:Q",
    y=alt.Y("estado_pedido:N", sort="-x"),
    text="cantidad_texto:N",
)
grafico_estados = (barras_estados + texto_estados).properties(
    width=360,
    height=300,
    title=alt.TitleParams(text="Pedidos por estado"),
)

linea_meses = alt.Chart(pedidos_por_mes).mark_line(
    color=COLOR_PRINCIPAL, strokeWidth=2.4
).encode(
    x=alt.X("mes:T", title="Mes de compra", axis=alt.Axis(format="%Y-%m", labelAngle=-45, tickCount="month")),
    y=alt.Y("cantidad_pedidos:Q", title="Cantidad de pedidos", scale=alt.Scale(zero=True)),
    tooltip=[
        alt.Tooltip("mes:T", title="Mes", format="%Y-%m"),
        alt.Tooltip("cantidad_texto:N", title="Pedidos"),
    ],
)
puntos_meses = alt.Chart(pedidos_por_mes).mark_point(
    filled=True, color=COLOR_PRINCIPAL, size=34
).encode(
    x="mes:T",
    y="cantidad_pedidos:Q",
    tooltip=[
        alt.Tooltip("mes:T", title="Mes", format="%Y-%m"),
        alt.Tooltip("cantidad_texto:N", title="Pedidos"),
    ],
)
grafico_meses = (linea_meses + puntos_meses).properties(
    width=610,
    height=300,
    title=alt.TitleParams(
        text="Pedidos registrados por mes",
        subtitle="Conteo de order_id distintos según la fecha de compra",
    ),
)

panel_pedidos = alt.hconcat(grafico_estados, grafico_meses, spacing=36).resolve_scale(y="independent")
estilo_profesional(panel_pedidos)

alt.HConcatChart(...)

# 5. Variables vinculadas con el costo de envío

## 5.1. Variable objetivo: `freight_value`

`freight_value` representa el costo de envío asignado a cada ítem del pedido. La distribución completa se conserva para los cálculos descriptivos. El histograma se limita al percentil 99 para que la concentración principal pueda observarse sin que los valores extremos compriman la escala horizontal.

In [12]:
freight = order_items["freight_value"].dropna()
percentiles_freight = freight.quantile([0.00, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00])
resumen_freight = pd.DataFrame(
    {
        "estadistico": ["cantidad", "media", "desvío estándar"],
        "valor": [freight.count(), freight.mean(), freight.std()],
    }
)
display(resumen_freight)
display(percentiles_freight.rename("freight_value").to_frame())

limite_freight = freight.quantile(0.99)
mediana_freight = freight.median()
conteos, bordes = np.histogram(freight[freight.le(limite_freight)], bins=35)
hist_freight = pd.DataFrame(
    {
        "desde": bordes[:-1],
        "hasta": bordes[1:],
        "cantidad": conteos,
    }
)
hist_freight["cantidad_texto"] = hist_freight["cantidad"].map(numero_es)

barras_freight = alt.Chart(hist_freight).mark_bar(
    color=COLOR_SECUNDARIO, opacity=0.9
).encode(
    x=alt.X("desde:Q", bin="binned", title="freight_value"),
    x2="hasta:Q",
    y=alt.Y("cantidad:Q", title="Cantidad de ítems"),
    tooltip=[
        alt.Tooltip("desde:Q", title="Desde", format=".2f"),
        alt.Tooltip("hasta:Q", title="Hasta", format=".2f"),
        alt.Tooltip("cantidad_texto:N", title="Ítems"),
    ],
)
regla_mediana = alt.Chart(pd.DataFrame({"mediana": [mediana_freight]})).mark_rule(
    color=COLOR_ACENTO, strokeWidth=2, strokeDash=[6, 4]
).encode(x="mediana:Q")
texto_mediana = alt.Chart(pd.DataFrame({"mediana": [mediana_freight], "etiqueta": [f"Mediana: {mediana_freight:.2f}"]})).mark_text(
    align="left", dx=7, dy=-135, color=COLOR_ACENTO, font="Arial", fontSize=10, fontWeight="bold"
).encode(x="mediana:Q", text="etiqueta:N")

grafico_freight = (barras_freight + regla_mediana + texto_mediana).properties(
    width=820,
    height=340,
    title=alt.TitleParams(
        text="Distribución de freight_value",
        subtitle=f"Visualización acotada al percentil 99 ({limite_freight:.2f}); las estadísticas incluyen todos los valores",
    ),
)
estilo_profesional(grafico_freight)

,estadistico,valor
0,cantidad,112650.000000
1,media,19.990320
2,desvío estándar,15.806405


,freight_value
0.00,0.000
0.25,13.080
0.50,16.260
0.75,21.150
0.90,34.041
0.95,45.120
0.99,84.520
1.00,409.680


alt.LayerChart(...)

## 5.2. Características físicas de los productos

La tabla `products` informa peso, largo, alto y ancho. El volumen se calcula como el producto de las tres dimensiones y se expresa en centímetros cúbicos. Estas variables se conservan como posibles predictores; sus asociaciones con `freight_value` se evaluarán en análisis posteriores.

In [13]:
products["product_volume_cm3"] = (
    products["product_length_cm"]
    * products["product_height_cm"]
    * products["product_width_cm"]
)

variables_fisicas = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_volume_cm3",
]

resumen_fisico = products[variables_fisicas].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T
resumen_fisico["nulos"] = products[variables_fisicas].isna().sum()
display(resumen_fisico)


def preparar_histograma(serie, bins=30):
    valores = serie.dropna()
    limite = valores.quantile(0.99)
    conteos, bordes = np.histogram(valores[valores.le(limite)], bins=bins)
    datos = pd.DataFrame(
        {"desde": bordes[:-1], "hasta": bordes[1:], "cantidad": conteos}
    )
    datos["cantidad_texto"] = datos["cantidad"].map(numero_es)
    return datos, limite


hist_peso, limite_peso = preparar_histograma(products["product_weight_g"])
hist_volumen, limite_volumen = preparar_histograma(products["product_volume_cm3"])


def grafico_histograma_fisico(datos, titulo, subtitulo, eje_x, color):
    return alt.Chart(datos).mark_bar(color=color, opacity=0.9).encode(
        x=alt.X("desde:Q", bin="binned", title=eje_x),
        x2="hasta:Q",
        y=alt.Y("cantidad:Q", title="Cantidad de productos"),
        tooltip=[
            alt.Tooltip("desde:Q", title="Desde", format=".1f"),
            alt.Tooltip("hasta:Q", title="Hasta", format=".1f"),
            alt.Tooltip("cantidad_texto:N", title="Productos"),
        ],
    ).properties(
        width=450,
        height=300,
        title=alt.TitleParams(text=titulo, subtitle=subtitulo),
    )


grafico_peso = grafico_histograma_fisico(
    hist_peso,
    "Distribución del peso",
    f"Visualización hasta el percentil 99 ({limite_peso:.1f} g)",
    "Peso del producto (g)",
    COLOR_SECUNDARIO,
)
grafico_volumen = grafico_histograma_fisico(
    hist_volumen,
    "Distribución del volumen",
    f"Visualización hasta el percentil 99 ({limite_volumen:.1f} cm³)",
    "Volumen del producto (cm³)",
    COLOR_VERDE,
)

panel_fisico = alt.hconcat(grafico_peso, grafico_volumen, spacing=36).resolve_scale(
    x="independent", y="independent"
)
estilo_profesional(panel_fisico)

,count,mean,std,min,25%,50%,75%,90%,95%,99%,max,nulos
product_weight_g,32949.0,2276.472488,4282.038731,0.0,300.0,700.0,1900.0,6300.0,10850.0,22538.00,40425.0,2
product_length_cm,32949.0,30.815078,16.914458,7.0,18.0,25.0,38.0,53.0,65.0,100.00,105.0,2
product_height_cm,32949.0,16.937661,13.637554,2.0,8.0,13.0,21.0,33.0,44.0,69.00,105.0,2
product_width_cm,32949.0,23.196728,12.079047,6.0,15.0,20.0,30.0,40.0,47.0,63.00,118.0,2
product_volume_cm3,32949.0,16564.096695,27057.041650,168.0,2880.0,6840.0,18480.0,41238.8,63369.0,135917.44,296208.0,2


alt.HConcatChart(...)

# 6. Construcción de la muestra analítica

En este notebook la unidad de análisis se define como un pedido con estado `delivered` y un único ítem. A partir de esa definición se incorporan los datos del cliente, el vendedor, el producto y las coordenadas asociadas a sus prefijos postales.

El número de registros resultante de cada condición se informa por separado.

In [14]:
cantidad_items = order_items.groupby("order_id").size().rename("cantidad_items")
orders_con_items = orders.merge(
    cantidad_items, left_on="order_id", right_index=True, how="left", validate="one_to_one"
)

pedidos_entregados = orders_con_items.loc[orders_con_items["order_status"].eq("delivered")].copy()
pedidos_un_item = pedidos_entregados.loc[pedidos_entregados["cantidad_items"].eq(1)].copy()
items_un_item = order_items.loc[order_items["order_id"].isin(pedidos_un_item["order_id"])].copy()

muestra = (
    pedidos_un_item
    .merge(customers, on="customer_id", how="left", validate="many_to_one")
    .merge(items_un_item, on="order_id", how="left", validate="one_to_one")
    .merge(sellers, on="seller_id", how="left", validate="many_to_one")
    .merge(products, on="product_id", how="left", validate="many_to_one")
)

geo_por_prefijo = (
    tablas["geolocation"]
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        latitud=("geolocation_lat", "median"),
        longitud=("geolocation_lng", "median"),
    )
)

geo_cliente = geo_por_prefijo.rename(
    columns={"geolocation_zip_code_prefix": "customer_geo_zip", "latitud": "customer_lat", "longitud": "customer_lng"}
)
geo_vendedor = geo_por_prefijo.rename(
    columns={"geolocation_zip_code_prefix": "seller_geo_zip", "latitud": "seller_lat", "longitud": "seller_lng"}
)

muestra = (
    muestra
    .merge(geo_cliente, left_on="customer_zip_code_prefix", right_on="customer_geo_zip", how="left", validate="many_to_one")
    .merge(geo_vendedor, left_on="seller_zip_code_prefix", right_on="seller_geo_zip", how="left", validate="many_to_one")
)

muestra["geografia_completa"] = muestra[["customer_lat", "customer_lng", "seller_lat", "seller_lng"]].notna().all(axis=1)
muestra["atributos_fisicos_completos"] = muestra[
    ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
].notna().all(axis=1)

muestra_geo = muestra.loc[muestra["geografia_completa"]].copy()
muestra_final = muestra_geo.loc[muestra_geo["atributos_fisicos_completos"]].copy()

flujo_muestra = pd.DataFrame(
    {
        "etapa": [
            "Pedidos originales",
            "Pedidos entregados",
            "Entregados con un ítem",
            "Con geografía completa",
            "Con atributos físicos completos",
        ],
        "cantidad": [
            len(orders),
            len(pedidos_entregados),
            len(muestra),
            len(muestra_geo),
            len(muestra_final),
        ],
        "orden": [1, 2, 3, 4, 5],
    }
)
flujo_muestra["porcentaje_inicial"] = 100 * flujo_muestra["cantidad"] / len(orders)
flujo_muestra["cantidad_texto"] = flujo_muestra["cantidad"].map(numero_es)
flujo_muestra["porcentaje_texto"] = flujo_muestra["porcentaje_inicial"].map(lambda x: f"{x:.2f}%")
flujo_muestra["etiqueta"] = flujo_muestra["cantidad_texto"] + "  (" + flujo_muestra["porcentaje_texto"] + ")"
display(flujo_muestra[["etapa", "cantidad", "porcentaje_inicial"]])

barras_flujo = alt.Chart(flujo_muestra).mark_bar(
    color=COLOR_PRINCIPAL, cornerRadiusEnd=4, size=24
).encode(
    x=alt.X(
        "cantidad:Q",
        title="Cantidad de pedidos",
        scale=alt.Scale(domain=[0, len(orders) * 1.14]),
    ),
    y=alt.Y(
        "etapa:N",
        sort=alt.SortField("orden", order="ascending"),
        title=None,
        axis=alt.Axis(labelLimit=260),
    ),
    tooltip=[
        "etapa:N",
        alt.Tooltip("cantidad_texto:N", title="Pedidos"),
        alt.Tooltip("porcentaje_texto:N", title="Sobre pedidos originales"),
    ],
)
etiquetas_flujo = alt.Chart(flujo_muestra).mark_text(
    align="left", dx=7, font="Arial", fontSize=10, color=COLOR_TEXTO
).encode(
    x="cantidad:Q",
    y=alt.Y("etapa:N", sort=alt.SortField("orden", order="ascending")),
    text="etiqueta:N",
)

grafico_flujo = (barras_flujo + etiquetas_flujo).properties(
    width=790,
    height=280,
    title=alt.TitleParams(
        text="Construcción de la muestra analítica",
        subtitle="Entre paréntesis se informa el porcentaje respecto de los pedidos originales",
    ),
)
estilo_profesional(grafico_flujo)

,etapa,cantidad,porcentaje_inicial
0,Pedidos originales,99441,100.000000
1,Pedidos entregados,96478,97.020344
2,Entregados con un ítem,86843,87.331181
3,Con geografía completa,86411,86.896753
4,Con atributos físicos completos,86397,86.882674


alt.LayerChart(...)

## 6.1. Casos sin cobertura geográfica

La cobertura geográfica se controla sobre los pedidos entregados con un único ítem. Se considera completa cuando están disponibles la latitud y la longitud tanto del cliente como del vendedor.

In [15]:
sin_geografia = int((~muestra["geografia_completa"]).sum())
porcentaje_sin_geografia = 100 * sin_geografia / len(muestra)

resumen_exclusion = pd.DataFrame(
    {
        "concepto": ["Muestra antes del control geográfico", "Casos sin geografía completa", "Casos conservados"],
        "cantidad": [len(muestra), sin_geografia, len(muestra_geo)],
        "porcentaje_sobre_muestra": [100.0, porcentaje_sin_geografia, 100 - porcentaje_sin_geografia],
    }
)
display(resumen_exclusion)

print(
    f"Casos sin coordenadas completas: {sin_geografia:,} "
    f"({porcentaje_sin_geografia:.3f}% de los pedidos entregados con un único ítem)."
)
print(
    f"Casos con coordenadas completas: {len(muestra_geo):,} "
    f"({100 - porcentaje_sin_geografia:.3f}% de la misma muestra)."
)
print("El filtro se aplica a la muestra analítica; los archivos de origen se conservan sin modificaciones.")

,concepto,cantidad,porcentaje_sobre_muestra
0,Muestra antes del control geográfico,86843,100.000000
1,Casos sin geografía completa,432,0.497449
2,Casos conservados,86411,99.502551


Casos sin coordenadas completas: 432 (0.497% de los pedidos entregados con un único ítem).
Casos con coordenadas completas: 86,411 (99.503% de la misma muestra).
El filtro se aplica a la muestra analítica; los archivos de origen se conservan sin modificaciones.


# 7. Síntesis del análisis inicial

El notebook documentó las nueve tablas, sus atributos y las columnas utilizadas para vincularlas. También presentó sus dimensiones, valores faltantes y conteos principales, y describió las distribuciones de `freight_value`, peso y volumen.

La construcción expuesta parte de 99.441 pedidos y aplica, de forma secuencial, las condiciones definidas para la muestra analítica. Los resultados de cada etapa se informan en la sección 6. Este análisis es descriptivo: no establece relaciones causales ni evalúa todavía el desempeño de modelos predictivos.